# Capture delay と word の探索・確認

QuEL-1 の `capture_delay`（128 ns刻み）と `capture_delay_word`（1 word = 8 ns、指定は4 words刻み）を組み合わせ、パルス全体の中点をキャプチャ窓の中央に配置し、前後32 ns以上の余白を確保します。API名は単数形の **`capture_delay_word`** です。

各測定後に遅延設定を復元します。設定ファイルへの保存は `save_params()` を明示的に呼んだ場合だけ行います。解析・描画は [`capture_delay_tools`](capture_delay_tools/README.md) にまとめています。ノートブックの実行環境に `.venv` を選び、このリポジトリのルートを作業ディレクトリにしてください。環境構築は [README](README.md) を参照してください。QuEL-3はword指定に対応していません。

波形は `calibrator.waveforms` に保存され、各 `Waveform.iq` は `time_ns` 座標付きの `xarray.DataArray` です。解析結果・候補一覧は pandas DataFrame です。例えば `calibrator.waveforms[0].iq.sel(time_ns=slice(100, 400))` で時間範囲を選択できます。


## 1. 装置の準備

In [ ]:
import os
from pathlib import Path

import qubex as qb

from capture_delay_tools import CaptureDelayCalibrator

# 環境変数で各環境の設定リポジトリを指定できます。
config_root = (
    Path(os.environ.get("QUBEX_CONFIG_ROOT", "./tmp/qubex-config"))
    .expanduser()
    .resolve()
)
system_id = "64Qv3"
exp = qb.Experiment(
    system_id=system_id,
    config_dir=config_root / system_id / "config",
    params_dir=config_root / system_id / "params",
    # muxes=[0,1,2,3,4,5,6,7,8,9,12,13,14,15],
    muxes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
)

In [ ]:
exp.connect()

## 2. 掃引

初期値は、`exp` に選択されているMUXについて、**delayを0〜18、wordを0に固定**した19点の粗い掃引です。MUX番号は実験の対象から取得します。掃引セルで両方の範囲を編集できます。

各MUX内でdelayとwordの全組み合わせを順番に測定します。例えばdelayが3点、wordが4点なら、対象数によらず MUX ごとに12回の同時取得です。wordのみを調べる場合は、delayを1値に固定し、`word_sweep` に `range(0, 16, 4)` などを指定してください。`capture_delay_word` を省略した場合は設定済みのword値を使用し、その値も結果に記録します。

|IQ|のヒートマップは、横軸が **delay/word の組**、縦軸がキャプチャ内の時刻（ns）、色が振幅です。横軸は測定点のカテゴリ表示で、隣接点の間隔が必ずしも同じns幅とは限りません。合計オフセットは `128 * delay + 8 * word` nsです。

平均回数は1024 shotsです。掃引・中央確認・最終確認に同じ回数を使用します。

図は qubit ごとに独立して表示します。`plot_final(show=False)` などは qubit 名をキーにした Figure の辞書を返します。

### キャプチャ遅延確認用の固定周波数

共鳴周波数の探索前に実行するため、`readout_frequencies_ghz` に MUX ごとの絶対周波数（GHz）を指定します。装置・配線・フィルタの通過帯域に合わせて設定してください。同じ条件の MUX には共通値を使えます。未指定の MUX は測定前にエラーにします。MUX 内の代表1対象だけ送信し、選択中の全対象を同時取得します。周波数の変更・復元は MUX ごとの掃引全体で1回にまとめ、掃引・再検証・最終確認で同じ周波数を使います。測定後は設定を戻し、周波数の YAML は変更しません。

In [ ]:
# 接続先の帯域に合わせて入力してください。例: {0: 9.0}（9 GHz）
# 9 GHz は記法の例であり、装置共通の推奨値ではありません。
readout_frequencies_ghz = {index: 11.0 for index in range(16)}
readout_frequencies_ghz

In [ ]:
system = exp.system_manager.experiment_system
mux_indices = sorted({system.get_mux_by_qubit(q).index for q in exp.qubit_labels})
if not mux_indices:
    raise ValueError("Experimentのmuxes/qubitsと設定を確認してください。対象が空です。")

delay_sweep = {mux: range(19) for mux in mux_indices}
word_sweep = {mux: [0] for mux in mux_indices}

# wordだけを掃引する例（delayの7は粗い掃引で確認した値に変更）:
# delay_sweep = {mux: [7] for mux in mux_indices}
# word_sweep = {mux: range(0, 16, 4) for mux in mux_indices}

calibrator = CaptureDelayCalibrator(
    exp,
    readout_frequencies_ghz=readout_frequencies_ghz,
    n_shots=1024,
    snr_threshold=5.0,
    smoothing=5,
)
display(calibrator.sweep(delay_sweep, capture_delay_word=word_sweep))
calibrator.plot_sweep()

## 3. エッジ解析と中点による候補選択

赤は立ち上がり、橙は立ち下がり、破線は窓中央です。両端が検出できればその中点、立ち上がりだけなら `rise + duration/2`、立ち下がりだけなら `fall - duration/2` を使います。duration は測定時に記録した読み出しパルス長（ns）です。全対象で異なる合計遅延の2点以上が64 ns以内で整合することを要求します。全対象を含む範囲を中央へ配置し、前後32 ns以上の余白を持つ粗い遅延と word を選びます。

In [ ]:
# 必要なら windows={"Q00": (0, 500)}。単位はns。
# 特定の組だけを切り出す場合: windows={("Q00", 7, 4): (100, 400)}
_ = calibrator.analyze()
display(calibrator.select())
calibrator.plot_analysis()

## 4. wordを0・4・8・12で実測

粗い掃引で選んだdelayを固定し、**wordを0・4・8・12で4点測定**します。初回のword=0固定の掃引と合わせて、各MUXで通常23回の測定です。粗い結果は `coarse_calibrator` に保存し、ここからの `calibrator` はword掃引の結果です。

同じdelayでwordを増やしたとき、信号の位置が32 ns刻みで移動するかをヒートマップで確認します。細かい掃引の解析結果にも到達時刻の整合性を要求し、その後に中央で再検証します。

図は qubit ごとに独立して表示します。`plot_final(show=False)` などは qubit 名をキーにした Figure の辞書を返します。

In [ ]:
coarse_calibrator = calibrator
if not coarse_calibrator.selected_delays:
    display(coarse_calibrator.selection)
    raise ValueError(
        "粗いdelayの候補がありません。掃引範囲とselectionのreasonを確認してください。"
    )

calibrator = CaptureDelayCalibrator(
    exp,
    readout_frequencies_ghz=readout_frequencies_ghz,
    n_shots=1024,
    snr_threshold=5.0,
    smoothing=5,
)
display(
    calibrator.sweep(
        {mux: [delay] for mux, delay in coarse_calibrator.selected_delays.items()},
        capture_delay_word={
            mux: range(0, 16, 4) for mux in coarse_calibrator.selected_delays
        },
    )
)
_ = calibrator.analyze()
display(calibrator.select())
calibrator.plot_analysis()

## 5. 中央で再測定

`selection` の `selected_delay` と `selected_capture_delay_word` を一時適用します。`midpoint_ns` は実測両端の中点、`error_ns` は中点から窓中央を引いた値です。片側からの推定で候補を選んだ場合も、ここでは両端が検出できることを確認します。

In [ ]:
display(calibrator.selected_delays)
display(calibrator.verify())
calibrator.plot_verification()

## 6. 両方の最終値への換算

全対象の両端が再測定で検出できた場合に限り、最も早い開始から最も遅い終了までの中点を窓中央へ合わせます。前後32 ns以上の余白を確保できる非負の32 ns刻みの候補から、中央に最も近い値を選びます。合計遅延を128 ns単位の `capture_delay` と残りの `capture_delay_word` に分解します。

`predicted_midpoint_ns` は予測中点、`predicted_pre_margin_ns` / `predicted_post_margin_ns` は最小予測余白です。片側しか確認できない場合や、`pulse_does_not_fit` の場合は採用しません。キャプチャ長・掃引範囲を見直してください。

In [ ]:
display(calibrator.finalize())
display(
    {
        "capture_delay": calibrator.final_delays,
        "capture_delay_word": calibrator.final_delay_words,
    }
)

## 7. 最終波形の確認

計算したdelayとwordを合計遅延（ns）に換算して再測定します。予測では両端に32 ns以上の余白があります。I/Q波形でも波形全体が収まっていることを確認してください。

`final_delays` が空の場合は `calibrator.final_report` の `reason` を確認します。`verification_failed` は必要な対象の検証失敗、`inconsistent_targets` は対象間の時刻差が大きすぎる状態です。掃引・解析・選択・検証をやり直すと、両方の最終値がクリアされます。その場合は検証後に `finalize()` を再実行してください。

詳細は `calibrator.analysis`、`calibrator.verification`、生波形は `calibrator.waveforms` に残ります。波形ごとに `delay`、`capture_delay_word`、`total_delay_ns` を記録しています。

補正後の図の見出しには適用したdelayとwordを表示します。取得済みの補正波形は `calibrator.plot_final()` で再測定せずに描画できます。

In [ ]:
# finalizeのセルを飛ばした場合も、検証済みデータから最終値を計算します。
if not calibrator.final_delays and not calibrator.verification.empty:
    display(calibrator.finalize())

if calibrator.final_delays:
    display(
        {
            "capture_delay": calibrator.final_delays,
            "capture_delay_word": calibrator.final_delay_words,
        }
    )
    calibrator.check_final()
else:
    display(calibrator.selection)
    display(calibrator.verification)
    display(calibrator.final_report)
    if calibrator.final_report.empty:
        print(
            "検証済みデータがありません。select()・verify()の結果を確認してください。"
        )
    else:
        reasons = {
            "verification_failed": "対象の立ち上がりを再検出できません。",
            "inconsistent_targets": "検証対象の到達時刻の差が64 nsを超えています。",
        }
        for row in calibrator.final_report.itertuples():
            print(f"MUX{row.mux}: {reasons.get(row.reason, row.reason)}")
    print("採用できる最終値がないため、補正値による測定は実行していません。")

新しい保存先では Experiment の `config_loader.params_path` にある同名ファイルをベースにします。
`source_dir="/path/to/source/params"` で読み込み元を指定できます。
上書き時はファイルごとに保存先の既存ファイルを優先します。
元ファイルがない場合は、対応する単位と `default: 0` を持つ YAML を作成します。
コメントと引用符は保持しますが、空白やインデントは再出力時に変わることがあります。

## 最終候補の保存（任意）

最終波形を確認してから、保存先の params ディレクトリを指定します。
以下は手動で実行する保存例です。

```python
calibrator.save_params("/path/to/params")
# 既存ファイルを更新する場合:
calibrator.save_params("/path/to/params", overwrite=True)
```

`capture_delay.yaml` と `capture_delay_word.yaml` に最終候補の MUX を保存します。
他の MUX、メタデータ、コメント、引用符を保持します。空白やインデントは変わることがあります。
詳細は [README](README.md#最終候補を-yaml-に保存する) を参照してください。


In [ ]:
calibrator.save_params("./tmp")

### 解析環境

`TimeOfFlightAnalysis` は、このリポジトリのPython 3.13環境にインストールした `sqe-analysis` を直接呼び出します。別プロセスや実行時の依存インストールは行いません。固定リビジョンは `pyproject.toml` と `uv.lock` を参照してください。


### 更新後の再実行

カーネルを再起動し、準備セルから新しい `calibrator` を作成してください。
